In [2]:
import os
import pandas as pd

# ============================================
# File paths
# ============================================
excel_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Guardant/EnRich_Guardant.xlsx"
genomic_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Guardant/Genomic_data_final.csv"
output_sheet = "Guardant_Data"

# ============================================
# Read Excel file
# ============================================
guardant_df = pd.read_excel(excel_file, dtype=str)
guardant_df.columns = guardant_df.columns.str.strip()

if "Guardant_testing" not in guardant_df.columns:
    raise KeyError("'Guardant_testing' column not found in EnRich_Guardant.xlsx")

# ============================================
# Read genomic CSV
# ============================================
genomic_df = pd.read_csv(genomic_file, dtype=str)
genomic_df.columns = genomic_df.columns.str.strip()

if "Guardant Health Accession ID" not in genomic_df.columns:
    raise KeyError("'Guardant Health Accession ID' column not found in Genomic_data_final.csv")

# ============================================
# Extract all Guardant IDs from Guardant_testing
# Handles values like: A1278007, A0899144
# ============================================
guardant_ids = (
    guardant_df["Guardant_testing"]
    .dropna()
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)

# Remove blank / invalid values
guardant_ids = guardant_ids[
    ~guardant_ids.isin(["", "nan", "None", "NA"])
]

guardant_ids = pd.unique(guardant_ids)

# ============================================
# Match with genomic data
# ============================================
genomic_df["Guardant Health Accession ID"] = genomic_df["Guardant Health Accession ID"].astype(str).str.strip()

matched_df = genomic_df[
    genomic_df["Guardant Health Accession ID"].isin(guardant_ids)
].copy()

# ============================================
# Save to another sheet in same Excel file
# ============================================
with pd.ExcelWriter(
    excel_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    matched_df.to_excel(writer, sheet_name=output_sheet, index=False)

# ============================================
# Print summary
# ============================================
print("Unique Guardant_testing IDs found:", len(guardant_ids))
print("Matched rows in Genomic_data_final.csv:", len(matched_df))
print(f"Saved matched data to sheet: {output_sheet}")
print("Output file:", excel_file)

Unique Guardant_testing IDs found: 94
Matched rows in Genomic_data_final.csv: 759
Saved matched data to sheet: Guardant_Data
Output file: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Guardant/EnRich_Guardant.xlsx
